# Deep Learning — Aula 1
## O que significa uma máquina aprender?

*Leitura: Prince, *Understanding Deep Learning*, capítulos 1 e 2.*

Esta nota de aula é também o roteiro da aula. As células marcadas com `# AO VIVO` são escritas durante o encontro; leia o texto ao redor delas como a explicação do que estamos fazendo e por quê.

Ao final desta aula você deve ser capaz de explicar o que é *next token prediction* e por que ela é um problema de aprendizado supervisionado, de descrever a ideia de um espaço latente aprendido, e de escrever, para a regressão linear, o modelo, a função de perda, o seu gradiente e o seu mínimo. Esta última parte é o que você vai precisar para abrir a semana 2, que começa pelo problema 2.3 do livro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (6, 4)
rng = np.random.default_rng(0)

## 1. Um modelo de linguagem é um classificador

Todo modelo que vamos estudar no curso tem a mesma forma: uma função $\mathbf{y} = \mathbf{f}[\mathbf{x}, \boldsymbol\phi]$ que leva uma entrada $\mathbf{x}$ a uma saída $\mathbf{y}$, controlada por parâmetros $\boldsymbol\phi$. Aprender é escolher $\boldsymbol\phi$ a partir de exemplos.

Um modelo de linguagem como o que está por trás do ChatGPT ou do Claude resolve um problema que, à primeira vista, parece muito distante disso: conversar. Mas o que ele faz, de fato, é muito mais modesto. Dado um trecho de texto, ele devolve uma distribuição de probabilidade sobre qual será o próximo pedaço de texto (o próximo *token*). Para gerar uma resposta, sorteamos um token dessa distribuição, acrescentamos ao texto e repetimos.

Repare no que isso significa: prever o próximo token é um problema de **classificação**, em que as classes são os tokens possíveis. E é um problema **supervisionado**, porque para cada contexto existe uma resposta certa. O detalhe que muda tudo é que a resposta certa vem de graça. Qualquer texto da internet, lido da esquerda para a direita, é um conjunto de pares (contexto, próximo token) já rotulado. Ninguém precisou anotar nada. Por isso se diz que o pré-treino é *auto-supervisionado*.

Um LLM moderno passa por três fases, e cada uma corresponde a um tipo de aprendizado do capítulo 1. No **pré-treino**, ele aprende a prever o próximo token em trilhões de tokens de texto (auto-supervisionado). No **ajuste por instruções**, ele vê exemplos de perguntas e boas respostas escritos por pessoas (supervisionado no sentido clássico). No **ajuste por preferências** (RLHF e variantes), ele gera respostas, recebe uma nota e é ajustado para obter notas maiores (aprendizado por reforço).

Para quem vem da engenharia elétrica, a ideia não é estranha: é **predição linear** de um sinal, o mesmo princípio do modelo autorregressivo e do LPC usado em codificação de voz, só que com um sinal de valores discretos e um preditor não linear enorme.

Vamos construir o modelo de linguagem mais simples possível: ele olha apenas para o caractere atual e prevê o próximo.

In [ ]:
import urllib.request, re

URL = "https://www.gutenberg.org/cache/epub/55752/pg55752.txt"  # Dom Casmurro, Machado de Assis
try:
    bruto = urllib.request.urlopen(URL, timeout=20).read().decode("utf-8")
    inicio = bruto.find("*** START"); fim = bruto.find("*** END")
    bruto = bruto[bruto.find("\n", inicio):fim]
except Exception as e:
    print("Download falhou, usando texto reserva:", e)
    bruto = ("a energia elétrica chega às casas por linhas de distribuição. "
             "o transformador reduz a tensão e o disjuntor protege o circuito. "
             "quando a corrente passa do limite, o disjuntor abre e a casa fica sem luz. ") * 200

texto = bruto.lower()
texto = re.sub(r"[^a-záàâãéêíóôõúüç ,.;!?\-]", " ", texto)  # só letras e pontuação simples
texto = re.sub(r"\s+", " ", texto).strip()
print(len(texto), "caracteres")
print(texto[5000:5300])

O primeiro passo é decidir o vocabulário. Aqui cada caractere é um token. (LLMs reais usam pedaços de palavras, com vocabulários de dezenas de milhares de tokens, mas a lógica é a mesma.) Depois contamos, para cada par de caracteres consecutivos $(a, b)$, quantas vezes $b$ aparece logo depois de $a$.

In [ ]:
# AO VIVO: acompanhe e escreva junto


Cada linha da matriz de contagens, normalizada, é uma distribuição de probabilidade sobre o próximo caractere. Isso já é um classificador: a entrada é o caractere atual, a saída é um vetor de $V$ probabilidades. Somamos 1 a todas as contagens para que nenhum par tenha probabilidade zero.

In [ ]:
# AO VIVO: acompanhe e escreva junto


Depois de "q" vem quase sempre "u". O modelo aprendeu isso sozinho, só contando. Para gerar texto, sorteamos o próximo caractere dessa distribuição, e repetimos.

In [ ]:
# AO VIVO: acompanhe e escreva junto


O resultado tem a "textura" do português (sílabas plausíveis, espaços nos lugares certos) e nenhum sentido. Com um caractere de contexto não dá para fazer melhor.

Como medir se um modelo é bom? Olhamos a probabilidade que ele atribui ao que realmente aconteceu no texto. Um bom modelo dá probabilidade alta ao caractere que de fato veio. Por conveniência numérica usamos o negativo do logaritmo e tiramos a média. Esta quantidade, a *log-verossimilhança negativa*, é a função de perda que vamos reencontrar no capítulo 5 sob o nome de entropia cruzada, e é exatamente a perda com que se treina um LLM.

In [ ]:
# AO VIVO: acompanhe e escreva junto


O modelo é bem melhor que sortear letras ao acaso, mas para usar mais contexto teríamos um problema sério. Com dois caracteres de contexto, a tabela teria $V^3$ entradas; com dez, $V^{11}$, mais números do que átomos que você conseguiria juntar. A maioria desses contextos nunca aparece no texto, e a tabela não tem como generalizar de um contexto para outro parecido.

É aqui que entra a ideia central do *deep learning*. Em vez de uma tabela, usamos uma função com parâmetros, que aprende a transformar a entrada numa representação interna onde contextos parecidos ficam próximos uns dos outros. Essa representação é o **espaço latente**.

## 2. Espaço latente: o que uma transformação aprendida captura

Vamos olhar um espaço latente pronto. O GloVe atribui a cada palavra do inglês um vetor de 50 números, aprendido a partir de quais palavras aparecem perto de quais outras em um corpus enorme. Ninguém disse ao algoritmo o que é gênero, realeza ou geografia.

In [ ]:
!pip -q install gensim
import gensim.downloader as api
wv = api.load("glove-wiki-gigaword-50")   # ~66 MB; se der erro de numpy, reinicie a sessão e rode de novo
E = wv.vectors / np.linalg.norm(wv.vectors, axis=1, keepdims=True)   # vetores unitários

Se o espaço captura relações, então a diferença entre "king" e "man" deveria ser parecida com a diferença entre "queen" e "woman". Ou seja, $\text{king} - \text{man} + \text{woman}$ deveria cair perto de "queen". Medimos "perto" pelo cosseno do ângulo entre vetores.

In [ ]:
# AO VIVO: acompanhe e escreva junto


Note o argumento `excluir`. A versão popular desta demonstração remove da busca as três palavras de entrada. Sem essa remoção, o resultado muda bastante.

In [ ]:
analogia("man", "king", "woman", excluir=False)

A estrutura existe, mas é aproximada: o vetor resultante continua mais perto de "king" do que de "queen". Vale guardar a lição metodológica junto com a bonita. Uma visualização ajuda: projetamos alguns pares de palavras nas duas direções de maior variância (PCA) e ligamos cada par com uma seta.

In [ ]:
pares = [("man", "woman"), ("king", "queen"), ("uncle", "aunt"), ("brother", "sister"),
         ("france", "paris"), ("italy", "rome"), ("japan", "tokyo"), ("brazil", "brasilia")]
palavras = [w for par in pares for w in par]
X = np.array([vec(w) for w in palavras])
X = X - X.mean(axis=0)
_, _, Vt = np.linalg.svd(X, full_matrices=False)
Z = X @ Vt[:2].T

for i in range(0, len(palavras), 2):
    plt.annotate("", Z[i + 1], Z[i], arrowprops=dict(arrowstyle="->", color="gray"))
for (z1, z2), w in zip(Z, palavras):
    plt.text(z1, z2, w)
plt.scatter(Z[:, 0], Z[:, 1], s=8)
plt.title("Pares de palavras projetados em 2D (PCA)")
plt.show()

As setas de gênero apontam numa direção parecida entre si, e as setas país→capital em outra. Uma relação semântica virou uma direção no espaço. Esse é o tipo de coisa que uma rede profunda faz em cada camada: aprende uma transformação que leva os dados a um espaço onde o problema fica mais simples. O resto do curso é sobre como essas transformações são construídas e treinadas. Começamos pelo caso mais simples de todos.

## 3. Aprendizado supervisionado: o caso da reta

Considere a calibração de um sensor de temperatura. Colocamos o sensor junto a um termômetro de referência em várias temperaturas e anotamos a tensão de saída. Queremos um modelo que, dada a leitura de tensão $x$, devolva a temperatura $y$.

Na notação do capítulo 2, o modelo é a reta
$$y = f[x, \boldsymbol\phi] = \phi_0 + \phi_1 x,$$
e os dados de treino são os pares $\{x_i, y_i\}_{i=1}^{I}$.

In [ ]:
rng = np.random.default_rng(42)
I = 20
y = rng.uniform(10, 90, I)                          # temperatura de referência (°C)
x = 0.01 * y + 0.05 + rng.normal(0, 0.1, I)         # leitura do sensor (V), com ruído

plt.scatter(x, y)
plt.xlabel("leitura x (V)"); plt.ylabel("temperatura y (°C)")
plt.show()

Precisamos de uma forma de dizer quanto um par de parâmetros é ruim. A escolha mais comum é a soma dos quadrados dos erros, equação 2.5 do livro:
$$L[\boldsymbol\phi] = \sum_{i=1}^{I} \left( f[x_i, \boldsymbol\phi] - y_i \right)^2 = \sum_{i=1}^{I} \left( \phi_0 + \phi_1 x_i - y_i \right)^2.$$
Aprender, agora, tem um significado preciso: encontrar $\hat{\boldsymbol\phi} = \operatorname{argmin}_{\boldsymbol\phi} L[\boldsymbol\phi]$. Escrevemos o modelo e a perda em código.

In [ ]:
# AO VIVO: acompanhe e escreva junto


Como só há dois parâmetros, podemos simplesmente calcular a perda para uma grade de valores e olhar a superfície inteira. Isso nunca será possível com uma rede de verdade, que tem milhões de parâmetros, mas aqui ajuda a criar a intuição.

In [ ]:
# AO VIVO: acompanhe e escreva junto


A superfície é um vale alongado. Para descer até o fundo sem ver a superfície inteira, fazemos o que faria alguém perdido numa montanha com neblina: medimos para que lado o terreno desce e damos um pequeno passo nessa direção. O lado que mais sobe é o gradiente $\partial L / \partial \boldsymbol\phi$; andamos no sentido oposto:
$$\boldsymbol\phi \leftarrow \boldsymbol\phi - \alpha \frac{\partial L}{\partial \boldsymbol\phi}.$$

Ainda não calculamos o gradiente (isso é o seu exercício). Por enquanto, vamos estimá-lo como qualquer engenheiro faria: perturbando cada parâmetro um pouquinho e medindo a variação da perda (diferença central).

In [ ]:
# AO VIVO: acompanhe e escreva junto


In [ ]:
plt.contour(p0, p1, np.log(L), levels=30)
plt.plot(traj[:, 0], traj[:, 1], ".-", color="red", ms=2)
plt.xlabel(r"$\phi_0$"); plt.ylabel(r"$\phi_1$")
plt.show()

O caminho desce rápido até o fundo do vale e depois anda devagar ao longo dele. Esse comportamento, que vem do formato alongado da superfície, vai reaparecer quando estudarmos otimização no capítulo 6. Experimente aumentar `alpha` para 0.05 e veja o que acontece.

## 4. Exercício

Em dupla, no papel:

**(a) Problema 2.1.** Calcule as expressões de $\partial L/\partial\phi_0$ e $\partial L/\partial\phi_1$ para a perda da equação 2.5.

**(b) Problema 2.2.** Iguale as duas derivadas a zero e resolva para $\phi_0$ e $\phi_1$. Você deve chegar a uma fórmula fechada para a melhor reta.

Depois, no notebook, implemente suas respostas nas células abaixo. O gradiente analítico deve coincidir com o numérico em qualquer ponto, e a solução fechada deve coincidir com o ponto onde o gradiente descendente parou.

In [ ]:
def grad_analitico(phi):
    # sua resposta do problema 2.1
    return np.array([..., ...])

def phi_fechado(x, y):
    # sua resposta do problema 2.2
    phi0 = ...
    phi1 = ...
    return np.array([phi0, phi1])

In [ ]:
teste = np.array([3.0, 50.0])
print("gradiente numérico: ", grad_numerico(teste))
print("gradiente analítico:", grad_analitico(teste))
print()
print("gradiente descendente:", traj[-1])
print("solução fechada:      ", phi_fechado(x, y))

Para a regressão linear a solução fechada existe, e seria tolice usar gradiente descendente. Para uma rede profunda, a equação "gradiente igual a zero" não tem solução que se possa escrever, e o único caminho é o iterativo. Calcular o gradiente, porém, continua sendo possível de forma exata e automática, e é isso que o PyTorch faz por baixo. Se quiser uma prévia:

In [ ]:
import torch
xt, yt = torch.tensor(x), torch.tensor(y)
phi_t = torch.tensor([3.0, 50.0], dtype=torch.float64, requires_grad=True)
L_t = torch.sum((phi_t[0] + phi_t[1] * xt - yt) ** 2)
L_t.backward()
print(phi_t.grad)

## 5. Para onde olha a função de perda

Vamos desenhar a reta ajustada junto com os erros que a perda mede.

In [ ]:
# AO VIVO: acompanhe e escreva junto


Os segmentos vermelhos são os resíduos $f[x_i, \boldsymbol\phi] - y_i$. Todos são verticais: a perda só mede erros na variável de saída $y$. Guarde essa imagem.

O modelo que construímos é **discriminativo**: vai diretamente da observação $x$ para a grandeza que queremos $y$. A seção 2.2 do livro discute a alternativa **generativa**, que modela o sentido físico do processo, $x = g[y, \boldsymbol\phi]$: a temperatura causa a leitura. Na calibração de sensores esse é o caminho mais natural para um engenheiro, que ajusta a curva do sensor (tensão em função da temperatura) e depois a inverte para medir.

### Para a semana 2

Antes do próximo encontro, faça uma aposta e escreva uma justificativa de duas linhas: se ajustarmos o modelo generativo $x = \phi_0 + \phi_1 y$ por mínimos quadrados e depois invertermos para obter $y$ a partir de $x$, obteremos a mesma reta que obtivemos hoje?

A semana 2 começa resolvendo essa pergunta, que é o problema 2.3 do livro.